In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vaanyakapur/us-china-thailand-tariff-diversion-data/htsdata.csv
/kaggle/input/datasets/vaanyakapur/us-china-thailand-tariff-diversion-data/China Tariffs_2026HTSRev10.pdf
/kaggle/input/datasets/vaanyakapur/us-china-thailand-tariff-diversion-data/DataWeb-Query-Export (1).xlsx


# Tariff Whack-a-Mole - Mapping Supply Chain Leakage to Thailand

When a major economy places tariffs on a manufacturing superpower, trade does dissapear, rather it relocated. This notebook studies whether U.S. tariff pressure on China was followed by rising U.S. imports from Thailand in similar manufacturing sectors.

The project builds a sector-level trade diversion simulator using monthly U.S. import data from China and Thailand.

## Research Question

Which Thai manufacturing sectors appear to gain the most when U.S. tariff pressure on China rises?

## Economic Concept

This project is based on trade diversion. If imports from China become more expensive due to tariffs, firms may shift sourcing toward countries such as Thailand.

## Methodology and Data 

This notebook uses:

- monthly panel data
- tariff shock indicators
- sector-level HTS codes
- Thailand import share
- China-to-Thailand substitution ratios
- event-study visualization
- fixed-effects regression
- a custom Whack-a-Mole Index
- a forward-looking tariff simulator

In [2]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import statsmodels.formula.api as smf

from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/vaanyakapur/us-china-thailand-tariff-diversion-data/htsdata.csv
/kaggle/input/datasets/vaanyakapur/us-china-thailand-tariff-diversion-data/China Tariffs_2026HTSRev10.pdf
/kaggle/input/datasets/vaanyakapur/us-china-thailand-tariff-diversion-data/DataWeb-Query-Export (1).xlsx


# 1. Loading the Trade Data

The main dataset is the USITC DataWeb export file. It contains monthly U.S. imports for consumption from China and Thailand, separated by HTS sector.

The key outcome variable is Customs Value, which represents the dollar value of imports.

In [4]:
# Find the DataWeb Excel file automatically
dataweb_files = glob.glob("/kaggle/input/**/DataWeb-Query-Export (1).xlsx", recursive=True)

if len(dataweb_files) == 0:
    raise FileNotFoundError("Could not find DataWeb-Query-Export (1).xlsx. Check your Kaggle dataset input.")

dataweb_path = dataweb_files[0]
print("Using file:", dataweb_path)

# Load the useful sheet
trade_raw = pd.read_excel(dataweb_path, sheet_name="Customs Value")

print("Shape:", trade_raw.shape)
trade_raw.head()

Using file: /kaggle/input/datasets/vaanyakapur/us-china-thailand-tariff-diversion-data/DataWeb-Query-Export (1).xlsx
Shape: (2905, 7)


,Data Type,Country,Year,Month,HTS Number,Description,Customs Value
0,Customs Value,China,"2,017.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,300897234
1,Customs Value,China,"2,018.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,343007740
2,Customs Value,China,"2,019.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,283855793
3,Customs Value,China,"2,020.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,192236003
4,Customs Value,China,"2,021.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,387691085


In [5]:
trade = trade_raw.copy()

# Clean column names
trade.columns = [
    str(c).strip().lower().replace(" ", "_")
    for c in trade.columns
]

print(trade.columns.tolist())
trade.head()

['data_type', 'country', 'year', 'month', 'hts_number', 'description', 'customs_value']


,data_type,country,year,month,hts_number,description,customs_value
0,Customs Value,China,"2,017.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,300897234
1,Customs Value,China,"2,018.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,343007740
2,Customs Value,China,"2,019.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,283855793
3,Customs Value,China,"2,020.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,192236003
4,Customs Value,China,"2,021.000",1.000,40.000,RUBBER AND ARTICLES THEREOF,387691085


In [6]:
# Drop rows with missing core values
trade = trade.dropna(
    subset=["country", "year", "month", "hts_number", "customs_value"],
    how="any"
)

# Convert year, month, and HTS code
trade["year"] = pd.to_numeric(trade["year"], errors="coerce").astype(int)
trade["month"] = pd.to_numeric(trade["month"], errors="coerce").astype(int)
trade["hts2"] = pd.to_numeric(trade["hts_number"], errors="coerce").astype(int).astype(str).str.zfill(2)

# Clean customs value
trade["customs_value"] = (
    trade["customs_value"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
)

trade["customs_value"] = pd.to_numeric(trade["customs_value"], errors="coerce")

# Create monthly date
trade["date"] = pd.to_datetime(
    trade["year"].astype(str) + "-" + trade["month"].astype(str).str.zfill(2) + "-01"
)

trade["year_month"] = trade["date"].dt.to_period("M").astype(str)

# Convert to millions of dollars
trade["import_value_musd"] = trade["customs_value"] / 1_000_000

trade.head()

,data_type,country,year,month,hts_number,description,customs_value,hts2,date,year_month,import_value_musd
0,Customs Value,China,2017,1,40.000,RUBBER AND ARTICLES THEREOF,300897234,40,2017-01-01,2017-01,300.897
1,Customs Value,China,2018,1,40.000,RUBBER AND ARTICLES THEREOF,343007740,40,2018-01-01,2018-01,343.008
2,Customs Value,China,2019,1,40.000,RUBBER AND ARTICLES THEREOF,283855793,40,2019-01-01,2019-01,283.856
3,Customs Value,China,2020,1,40.000,RUBBER AND ARTICLES THEREOF,192236003,40,2020-01-01,2020-01,192.236
4,Customs Value,China,2021,1,40.000,RUBBER AND ARTICLES THEREOF,387691085,40,2021-01-01,2021-01,387.691


In [7]:
print("Rows:", len(trade))
print("Countries:", trade["country"].unique())
print("Years:", trade["year"].min(), "to", trade["year"].max())
print("HTS sectors:", sorted(trade["hts2"].unique()))

trade.groupby(["country", "hts2"]).size().head(20)

Rows: 2904
Countries: ['China' 'Thailand']
Years: 2015 to 2025
HTS sectors: ['39', '40', '61', '62', '64', '73', '84', '85', '87', '90', '94']


country   hts2
China     39      132
          40      132
          61      132
          62      132
          64      132
          73      132
          84      132
          85      132
          87      132
          90      132
          94      132
Thailand  39      132
          40      132
          61      132
          62      132
          64      132
          73      132
          84      132
          85      132
          87      132
dtype: int64

# 2. Sector Mapping

HTS codes are official product categories, but they are hard to read. This section maps each HTS chapter to a more understandable manufacturing sector.

The notebook also creates two researcher-coded scores:

- supply chain mobility: how easily production can relocate
- Thai relevance score: how plausible Thailand is as a substitute production base

These scores will be used later to build a tariff exposure intensity variable.

In [8]:
sector_map = {
    "39": ["Plastics", "Materials", 4, 4],
    "40": ["Rubber products", "Materials", 4, 5],
    "61": ["Knit apparel", "Textiles", 3, 3],
    "62": ["Woven apparel", "Textiles", 3, 3],
    "64": ["Footwear", "Light manufacturing", 3, 3],
    "73": ["Articles of steel", "Industrial inputs", 4, 4],
    "84": ["Machinery & computers", "Advanced manufacturing", 5, 5],
    "85": ["Electronics & electrical equipment", "Advanced manufacturing", 5, 5],
    "87": ["Vehicles & auto parts", "Automotive", 4, 5],
    "90": ["Medical/precision instruments", "Advanced manufacturing", 4, 4],
    "94": ["Furniture", "Light manufacturing", 4, 4]
}

sector_df = pd.DataFrame([
    {
        "hts2": code,
        "sector": values[0],
        "sector_group": values[1],
        "supply_chain_mobility": values[2],
        "thai_relevance_score": values[3]
    }
    for code, values in sector_map.items()
])

sector_df

,hts2,sector,sector_group,supply_chain_mobility,thai_relevance_score
0,39,Plastics,Materials,4,4
1,40,Rubber products,Materials,4,5
2,61,Knit apparel,Textiles,3,3
3,62,Woven apparel,Textiles,3,3
4,64,Footwear,Light manufacturing,3,3
5,73,Articles of steel,Industrial inputs,4,4
6,84,Machinery & computers,Advanced manufacturing,5,5
7,85,Electronics & electrical equipment,Advanced manufacturing,5,5
8,87,Vehicles & auto parts,Automotive,4,5
9,90,Medical/precision instruments,Advanced manufacturing,4,4


In [9]:
trade = trade.merge(sector_df, on="hts2", how="left")

trade[["country", "date", "hts2", "sector", "import_value_musd"]].head()

,country,date,hts2,sector,import_value_musd
0,China,2017-01-01,40,Rubber products,300.897
1,China,2018-01-01,40,Rubber products,343.008
2,China,2019-01-01,40,Rubber products,283.856
3,China,2020-01-01,40,Rubber products,192.236
4,China,2021-01-01,40,Rubber products,387.691


# 3. Building the Monthly China-Thailand Panel

The unit of analysis is:

HTS sector × month

For each sector-month, the notebook compares U.S. imports from China against U.S. imports from Thailand.

This allows us to calculate Thailand's share of China-plus-Thailand imports.

In [10]:
panel = trade.pivot_table(
    index=[
        "date",
        "year",
        "month",
        "year_month",
        "hts2",
        "sector",
        "sector_group",
        "supply_chain_mobility",
        "thai_relevance_score"
    ],
    columns="country",
    values="import_value_musd",
    aggfunc="sum"
).reset_index()

panel.columns.name = None

for col in ["China", "Thailand"]:
    if col not in panel.columns:
        panel[col] = 0

panel = panel.rename(columns={
    "China": "china_imports_musd",
    "Thailand": "thailand_imports_musd"
})

panel["total_china_thai_musd"] = (
    panel["china_imports_musd"] + panel["thailand_imports_musd"]
)

panel["thailand_share"] = (
    panel["thailand_imports_musd"] / panel["total_china_thai_musd"].replace(0, np.nan)
)

panel["china_share"] = (
    panel["china_imports_musd"] / panel["total_china_thai_musd"].replace(0, np.nan)
)

panel["log_thai_china_ratio"] = (
    np.log1p(panel["thailand_imports_musd"]) -
    np.log1p(panel["china_imports_musd"])
)

panel.head()

,date,year,month,year_month,hts2,sector,sector_group,supply_chain_mobility,thai_relevance_score,china_imports_musd,thailand_imports_musd,total_china_thai_musd,thailand_share,china_share,log_thai_china_ratio
0,2015-01-01,2015,1,2015-01,39,Plastics,Materials,4,4,"1,173.325",38.941,"1,212.266",0.032,0.968,-3.381
1,2015-01-01,2015,1,2015-01,40,Rubber products,Materials,4,5,396.472,164.818,561.289,0.294,0.706,-0.874
2,2015-01-01,2015,1,2015-01,61,Knit apparel,Textiles,3,3,"1,164.870",64.537,"1,229.407",0.052,0.948,-2.879
3,2015-01-01,2015,1,2015-01,62,Woven apparel,Textiles,3,3,"1,237.621",28.279,"1,265.900",0.022,0.978,-3.745
4,2015-01-01,2015,1,2015-01,64,Footwear,Light manufacturing,3,3,"1,502.768",7.756,"1,510.524",0.005,0.995,-5.146


# 4. Tariff Shock Timeline

The project uses the U.S.-China Section 301 tariff period as the main shock.

As this dataset is HTS 2-digit sector-level rather than HS6 or HS10 product-level, the tariff variable is a sector-level proxy rather than a perfect product-level tariff match.

The goal is to estimate whether sectors with higher supply-chain relocation potential show stronger Thailand gains after tariff pressure rises.

In [11]:
def tariff_phase(date):
    if date < pd.Timestamp("2018-07-01"):
        return "Pre-tariff baseline"
    elif date < pd.Timestamp("2019-05-01"):
        return "Initial Section 301 shock"
    elif date < pd.Timestamp("2020-03-01"):
        return "Escalated tariff period"
    elif date < pd.Timestamp("2022-01-01"):
        return "COVID supply-chain disruption"
    else:
        return "Post-shock adjustment"


def tariff_rate_proxy(date):
    """
    Simplified tariff pressure proxy.

    This is not a perfect product-level tariff rate.
    It is a sector-level shock proxy based on the Section 301 escalation period.
    """
    if date < pd.Timestamp("2018-07-01"):
        return 0.00
    elif date < pd.Timestamp("2018-09-01"):
        return 0.25
    elif date < pd.Timestamp("2019-05-01"):
        return 0.10
    else:
        return 0.25


panel["tariff_phase"] = panel["date"].apply(tariff_phase)
panel["post_tariff"] = (panel["date"] >= pd.Timestamp("2018-07-01")).astype(int)
panel["tariff_rate_proxy"] = panel["date"].apply(tariff_rate_proxy)

# Researcher-coded exposure intensity
panel["exposure_score"] = (
    panel["supply_chain_mobility"] + panel["thai_relevance_score"]
) / 10

panel["tariff_intensity"] = panel["tariff_rate_proxy"] * panel["exposure_score"]

panel[[
    "date",
    "sector",
    "tariff_phase",
    "tariff_rate_proxy",
    "exposure_score",
    "tariff_intensity"
]].head()

,date,sector,tariff_phase,tariff_rate_proxy,exposure_score,tariff_intensity
0,2015-01-01,Plastics,Pre-tariff baseline,0.000,0.800,0.000
1,2015-01-01,Rubber products,Pre-tariff baseline,0.000,0.900,0.000
2,2015-01-01,Knit apparel,Pre-tariff baseline,0.000,0.600,0.000
3,2015-01-01,Woven apparel,Pre-tariff baseline,0.000,0.600,0.000
4,2015-01-01,Footwear,Pre-tariff baseline,0.000,0.600,0.000


## Why use a tariff intensity proxy?

A perfect tariff model would match every HS6 or HS10 product to the exact Section 301 tariff list. This dataset is currently at HTS 2-digit sector level, so the notebook uses a sector-level tariff intensity proxy:

tariff intensity = tariff rate proxy × sector exposure score

This still creates a useful research design because it tests whether sectors with higher relocation potential show stronger Thailand gains after tariff pressure rises. Thus, the result should be interpreted as a sector-level substitution signal.

In [12]:
panel = panel.sort_values(["hts2", "date"])

# Year-over-year growth
for col in [
    "china_imports_musd",
    "thailand_imports_musd",
    "thailand_share",
    "log_thai_china_ratio"
]:
    panel[f"{col}_yoy_pct"] = panel.groupby("hts2")[col].pct_change(12) * 100

# Pre-tariff baseline: before July 2018
baseline = (
    panel[panel["date"] < pd.Timestamp("2018-07-01")]
    .groupby("hts2")
    .agg(
        baseline_thai=("thailand_imports_musd", "mean"),
        baseline_china=("china_imports_musd", "mean"),
        baseline_share=("thailand_share", "mean"),
        baseline_ratio=("log_thai_china_ratio", "mean")
    )
    .reset_index()
)

panel = panel.merge(baseline, on="hts2", how="left")

panel["thai_gain_vs_baseline_musd"] = (
    panel["thailand_imports_musd"] - panel["baseline_thai"]
)

panel["china_change_vs_baseline_musd"] = (
    panel["china_imports_musd"] - panel["baseline_china"]
)

panel["share_gain_vs_baseline"] = (
    panel["thailand_share"] - panel["baseline_share"]
)

panel["china_decline_positive_musd"] = (
    -panel["china_change_vs_baseline_musd"]
).clip(lower=0)

panel.head()

,date,year,month,year_month,hts2,sector,sector_group,supply_chain_mobility,thai_relevance_score,china_imports_musd,thailand_imports_musd,total_china_thai_musd,thailand_share,china_share,log_thai_china_ratio,tariff_phase,post_tariff,tariff_rate_proxy,exposure_score,tariff_intensity,china_imports_musd_yoy_pct,thailand_imports_musd_yoy_pct,thailand_share_yoy_pct,log_thai_china_ratio_yoy_pct,baseline_thai,baseline_china,baseline_share,baseline_ratio,thai_gain_vs_baseline_musd,china_change_vs_baseline_musd,share_gain_vs_baseline,china_decline_positive_musd
0,2015-01-01,2015,1,2015-01,39,Plastics,Materials,4,4,"1,173.325",38.941,"1,212.266",0.032,0.968,-3.381,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,46.601,"1,284.727",0.035,-3.299,-7.660,-111.401,-0.003,111.401
1,2015-02-01,2015,2,2015-02,39,Plastics,Materials,4,4,989.339,26.990,"1,016.329",0.027,0.973,-3.566,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,46.601,"1,284.727",0.035,-3.299,-19.610,-295.388,-0.009,295.388
2,2015-03-01,2015,3,2015-03,39,Plastics,Materials,4,4,"1,295.903",44.486,"1,340.389",0.033,0.967,-3.350,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,46.601,"1,284.727",0.035,-3.299,-2.114,11.176,-0.002,0.000
3,2015-04-01,2015,4,2015-04,39,Plastics,Materials,4,4,"1,149.515",53.463,"1,202.977",0.044,0.956,-3.050,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,46.601,"1,284.727",0.035,-3.299,6.862,-135.212,0.009,135.212
4,2015-05-01,2015,5,2015-05,39,Plastics,Materials,4,4,"1,291.836",46.439,"1,338.275",0.035,0.965,-3.305,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,46.601,"1,284.727",0.035,-3.299,-0.162,7.109,-0.000,0.000


# 5. Descriptive Trade Patterns

We first compare total U.S. imports from China and Thailand across the selected manufacturing sectors.

In [13]:
country_monthly = (
    trade.groupby(["date", "country"], as_index=False)["import_value_musd"]
    .sum()
)

fig = px.line(
    country_monthly,
    x="date",
    y="import_value_musd",
    color="country",
    title="U.S. Imports from China vs Thailand in Selected Manufacturing Sectors",
    labels={
        "date": "Month",
        "import_value_musd": "Customs Value, Million USD",
        "country": "Partner Country"
    }
)

# Safer replacement for fig.add_vline with Timestamp annotation
tariff_date = "2018-07-01"

fig.add_shape(
    type="line",
    x0=tariff_date,
    x1=tariff_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(dash="dash", width=2)
)

fig.add_annotation(
    x=tariff_date,
    y=1,
    xref="x",
    yref="paper",
    text="Section 301 tariff shock begins",
    showarrow=False,
    yanchor="bottom",
    xanchor="left"
)

fig.show()

In [14]:
country_monthly = (
    trade.groupby(["date", "country"], as_index=False)["import_value_musd"]
    .sum()
)

fig = px.line(
    country_monthly,
    x="date",
    y="import_value_musd",
    color="country",
    title="U.S. Imports from China vs Thailand in Selected Manufacturing Sectors",
    labels={
        "date": "Month",
        "import_value_musd": "Customs Value, Million USD",
        "country": "Partner Country"
    }
)

# Safer replacement for fig.add_vline with Timestamp annotation
tariff_date = "2018-07-01"

fig.add_shape(
    type="line",
    x0=tariff_date,
    x1=tariff_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(dash="dash", width=2)
)

fig.add_annotation(
    x=tariff_date,
    y=1,
    xref="x",
    yref="paper",
    text="Section 301 tariff shock begins",
    showarrow=False,
    yanchor="bottom",
    xanchor="left"
)

fig.show()

In [15]:
def add_vertical_marker(fig, x_value, label, xref="x"):
    """
    Adds a vertical dashed line and annotation to a Plotly figure.
    This avoids Plotly's add_vline Timestamp bug on Kaggle.
    """
    fig.add_shape(
        type="line",
        x0=x_value,
        x1=x_value,
        y0=0,
        y1=1,
        xref=xref,
        yref="paper",
        line=dict(dash="dash", width=2)
    )
    
    fig.add_annotation(
        x=x_value,
        y=1,
        xref=xref,
        yref="paper",
        text=label,
        showarrow=False,
        yanchor="bottom",
        xanchor="left"
    )
    
    return fig


TARIFF_DATE = "2018-07-01"

print("Safe chart helper loaded.")

Safe chart helper loaded.


In [16]:
fig = px.line(
    panel,
    x="date",
    y="thailand_share",
    color="sector",
    title="Thailand's Share of U.S. Imports Relative to China by Sector",
    labels={
        "date": "Month",
        "thailand_share": "Thailand Share of China + Thailand Imports",
        "sector": "Sector"
    }
)

fig = add_vertical_marker(
    fig,
    x_value=TARIFF_DATE,
    label="Section 301 tariff shock"
)

fig.show()

# 6. Heatmap of Thailand's Sector Share

A heatmap makes it easier to see which sectors became more Thailand-heavy over time.

Darker or higher values mean Thailand captured a larger share of U.S. imports relative to China in that sector.

In [17]:
heatmap_data = panel.pivot_table(
    index="sector",
    columns="year",
    values="thailand_share",
    aggfunc="mean"
)

fig = px.imshow(
    heatmap_data,
    text_auto=".2f",
    aspect="auto",
    title="Heatmap: Thailand Share of China + Thailand Imports by Sector and Year",
    labels={
        "x": "Year",
        "y": "Sector",
        "color": "Thailand Share"
    }
)

fig.show()

# 7. Before vs After Tariff Shock

This section compares Thailand's average import share before and after the July 2018 tariff shock.

This is a simple first test of whether Thailand gained relative position after tariffs increased on China.

In [18]:
before_after = panel.copy()

before_after["period"] = np.where(
    before_after["date"] < pd.Timestamp("2018-07-01"),
    "Before tariff shock",
    "After tariff shock"
)

before_after_summary = (
    before_after
    .groupby(["sector", "period"], as_index=False)
    .agg(
        avg_thailand_share=("thailand_share", "mean"),
        avg_thailand_imports_musd=("thailand_imports_musd", "mean"),
        avg_china_imports_musd=("china_imports_musd", "mean")
    )
)

fig = px.bar(
    before_after_summary,
    x="sector",
    y="avg_thailand_share",
    color="period",
    barmode="group",
    title="Before vs After: Thailand Import Share by Sector",
    labels={
        "sector": "Sector",
        "avg_thailand_share": "Average Thailand Share",
        "period": "Period"
    }
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

# 8. Baseline Gains

To measure trade diversion, the notebook compares each sector's monthly values against its pre-tariff baseline.

The pre-tariff baseline is the average value before July 2018.

In [19]:
panel = panel.sort_values(["hts2", "date"]).copy()

# Year-over-year growth
growth_cols = [
    "china_imports_musd",
    "thailand_imports_musd",
    "thailand_share",
    "log_thai_china_ratio"
]

for col in growth_cols:
    panel[f"{col}_yoy_pct"] = panel.groupby("hts2")[col].pct_change(12) * 100

# Pre-tariff baseline
baseline = (
    panel[panel["date"] < pd.Timestamp("2018-07-01")]
    .groupby("hts2")
    .agg(
        baseline_thai=("thailand_imports_musd", "mean"),
        baseline_china=("china_imports_musd", "mean"),
        baseline_share=("thailand_share", "mean"),
        baseline_ratio=("log_thai_china_ratio", "mean")
    )
    .reset_index()
)

# Remove old baseline columns if rerunning this cell
baseline_cols = [
    "baseline_thai",
    "baseline_china",
    "baseline_share",
    "baseline_ratio"
]

panel = panel.drop(columns=[c for c in baseline_cols if c in panel.columns], errors="ignore")
panel = panel.merge(baseline, on="hts2", how="left")

panel["thai_gain_vs_baseline_musd"] = (
    panel["thailand_imports_musd"] - panel["baseline_thai"]
)

panel["china_change_vs_baseline_musd"] = (
    panel["china_imports_musd"] - panel["baseline_china"]
)

panel["share_gain_vs_baseline"] = (
    panel["thailand_share"] - panel["baseline_share"]
)

panel["china_decline_positive_musd"] = (
    -panel["china_change_vs_baseline_musd"]
).clip(lower=0)

panel[[
    "date",
    "sector",
    "thailand_imports_musd",
    "china_imports_musd",
    "thai_gain_vs_baseline_musd",
    "china_change_vs_baseline_musd",
    "share_gain_vs_baseline"
]].head()

,date,sector,thailand_imports_musd,china_imports_musd,thai_gain_vs_baseline_musd,china_change_vs_baseline_musd,share_gain_vs_baseline
0,2015-01-01,Plastics,38.941,"1,173.325",-7.660,-111.401,-0.003
1,2015-02-01,Plastics,26.990,989.339,-19.610,-295.388,-0.009
2,2015-03-01,Plastics,44.486,"1,295.903",-2.114,11.176,-0.002
3,2015-04-01,Plastics,53.463,"1,149.515",6.862,-135.212,0.009
4,2015-05-01,Plastics,46.439,"1,291.836",-0.162,7.109,-0.000


# 9. The Whack-a-Mole Index

The Whack-a-Mole Index ranks sectors where Thailand appears to benefit most from China-facing tariff pressure.

The index combines:

1. Thailand import gains versus the pre-tariff baseline
2. Thailand share gains
3. China import decline
4. Thailand-to-China substitution ratio
5. sector exposure score

This creates a custom trade diversion indicator rather than relying on one simple chart.

In [20]:
def safe_minmax(series):
    """
    Converts a numeric series to a 0 to 1 scale.
    If the column is constant, it returns 0.5 for every row.
    """
    s = pd.to_numeric(series, errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)
    s = s.fillna(s.median())
    
    if s.max() == s.min():
        return pd.Series([0.5] * len(s), index=s.index)
    
    return (s - s.min()) / (s.max() - s.min())


post_panel = panel[panel["post_tariff"] == 1].copy()

leaderboard = (
    post_panel
    .groupby(["hts2", "sector", "sector_group"], as_index=False)
    .agg(
        avg_thai_gain=("thai_gain_vs_baseline_musd", "mean"),
        total_thai_gain=("thai_gain_vs_baseline_musd", "sum"),
        avg_share_gain=("share_gain_vs_baseline", "mean"),
        avg_china_decline=("china_decline_positive_musd", "mean"),
        avg_log_ratio=("log_thai_china_ratio", "mean"),
        exposure_score=("exposure_score", "mean")
    )
)

# Normalize each component
leaderboard["avg_thai_gain_norm"] = safe_minmax(leaderboard["avg_thai_gain"])
leaderboard["avg_share_gain_norm"] = safe_minmax(leaderboard["avg_share_gain"])
leaderboard["avg_china_decline_norm"] = safe_minmax(leaderboard["avg_china_decline"])
leaderboard["avg_log_ratio_norm"] = safe_minmax(leaderboard["avg_log_ratio"])
leaderboard["exposure_score_norm"] = safe_minmax(leaderboard["exposure_score"])

# Custom weighted index
leaderboard["whack_a_mole_index"] = 100 * (
    0.30 * leaderboard["avg_thai_gain_norm"] +
    0.25 * leaderboard["avg_share_gain_norm"] +
    0.20 * leaderboard["avg_china_decline_norm"] +
    0.15 * leaderboard["avg_log_ratio_norm"] +
    0.10 * leaderboard["exposure_score_norm"]
)

leaderboard = leaderboard.sort_values("whack_a_mole_index", ascending=False)

leaderboard[[
    "hts2",
    "sector",
    "sector_group",
    "whack_a_mole_index",
    "avg_thai_gain",
    "avg_share_gain",
    "avg_china_decline",
    "exposure_score"
]]

,hts2,sector,sector_group,whack_a_mole_index,avg_thai_gain,avg_share_gain,avg_china_decline,exposure_score
7,85,Electronics & electrical equipment,Advanced manufacturing,75.391,721.588,0.073,"1,741.927",1.000
6,84,Machinery & computers,Advanced manufacturing,70.911,592.521,0.088,"1,602.739",1.000
1,40,Rubber products,Materials,55.836,182.719,0.212,79.707,0.900
8,87,Vehicles & auto parts,Automotive,23.600,74.788,0.049,67.400,0.900
10,94,Furniture,Light manufacturing,22.629,55.290,0.038,681.922,0.800
9,90,Medical/precision instruments,Advanced manufacturing,19.041,48.959,0.042,48.518,0.800
5,73,Articles of steel,Industrial inputs,17.201,39.746,0.036,57.238,0.800
0,39,Plastics,Materials,13.667,45.341,0.021,28.865,0.800
3,62,Woven apparel,Textiles,10.193,4.347,0.020,407.107,0.600
2,61,Knit apparel,Textiles,9.534,-4.718,0.012,345.095,0.600


In [21]:
fig = px.bar(
    leaderboard,
    x="whack_a_mole_index",
    y="sector",
    orientation="h",
    title="Tariff Whack-a-Mole Leaderboard: Thailand's Strongest Substitution Sectors",
    labels={
        "whack_a_mole_index": "Whack-a-Mole Index",
        "sector": "Sector"
    },
    hover_data=[
        "avg_thai_gain",
        "avg_share_gain",
        "avg_china_decline",
        "exposure_score"
    ]
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

# 10. Event Study

An event study checks the timing of Thailand's gains around the tariff shock. The event month is July 2018, when the first major Section 301 tariff shock began. A strong trade diversion story should show Thailand gaining after the shock, not randomly before it.

In [22]:
panel["months_since_tariff"] = (
    (panel["date"].dt.year - 2018) * 12 +
    (panel["date"].dt.month - 7)
)

event_window = panel[
    (panel["months_since_tariff"] >= -36) &
    (panel["months_since_tariff"] <= 89)
].copy()

event_summary = (
    event_window
    .groupby("months_since_tariff", as_index=False)
    .agg(
        mean_share_gain=("share_gain_vs_baseline", "mean"),
        mean_thai_gain=("thai_gain_vs_baseline_musd", "mean"),
        mean_log_ratio=("log_thai_china_ratio", "mean")
    )
)

fig = px.line(
    event_summary,
    x="months_since_tariff",
    y="mean_share_gain",
    markers=True,
    title="Event Study: Average Thailand Share Gain Around the 2018 Tariff Shock",
    labels={
        "months_since_tariff": "Months Since July 2018 Tariff Shock",
        "mean_share_gain": "Average Thailand Share Gain vs Baseline"
    }
)

# Safe vertical marker for numeric x-axis
fig.add_shape(
    type="line",
    x0=0,
    x1=0,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(dash="dash", width=2)
)

fig.add_annotation(
    x=0,
    y=1,
    xref="x",
    yref="paper",
    text="Tariff shock",
    showarrow=False,
    yanchor="bottom",
    xanchor="left"
)

fig.show()

In [23]:
event_sector = (
    event_window
    .groupby(["months_since_tariff", "sector"], as_index=False)
    .agg(mean_share_gain=("share_gain_vs_baseline", "mean"))
)

fig = px.line(
    event_sector,
    x="months_since_tariff",
    y="mean_share_gain",
    color="sector",
    title="Event Study by Sector: Thailand Share Gain Around Tariff Shock",
    labels={
        "months_since_tariff": "Months Since July 2018",
        "mean_share_gain": "Thailand Share Gain vs Baseline",
        "sector": "Sector"
    }
)

fig.add_shape(
    type="line",
    x0=0,
    x1=0,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(dash="dash", width=2)
)

fig.add_annotation(
    x=0,
    y=1,
    xref="x",
    yref="paper",
    text="Tariff shock",
    showarrow=False,
    yanchor="bottom",
    xanchor="left"
)

fig.show()

# 11. Fixed-Effects Regression

The goal is to estimate whether tariff intensity is associated with Thailand gaining relative to China.

The main dependent variable is:

log(Thailand imports + 1) - log(China imports + 1)

This is a log Thailand-to-China import ratio.

A positive coefficient means Thailand improved relative to China.

In [24]:
reg = panel.copy()

reg = reg.replace([np.inf, -np.inf], np.nan)

reg = reg.dropna(
    subset=[
        "log_thai_china_ratio",
        "tariff_intensity",
        "thailand_share",
        "thai_gain_vs_baseline_musd"
    ]
)

# Numeric monthly time trend
reg["time_index"] = (
    (reg["date"].dt.year - reg["date"].dt.year.min()) * 12 +
    reg["date"].dt.month
)

# A readable year-month fixed effect if needed later
reg["ym_fe"] = reg["date"].dt.to_period("M").astype(str)

print("Regression rows:", len(reg))
print("Sectors:", reg["hts2"].nunique())
print("Months:", reg["date"].nunique())

reg.head()

Regression rows: 1452
Sectors: 11
Months: 132


,date,year,month,year_month,hts2,sector,sector_group,supply_chain_mobility,thai_relevance_score,china_imports_musd,thailand_imports_musd,total_china_thai_musd,thailand_share,china_share,log_thai_china_ratio,tariff_phase,post_tariff,tariff_rate_proxy,exposure_score,tariff_intensity,china_imports_musd_yoy_pct,thailand_imports_musd_yoy_pct,thailand_share_yoy_pct,log_thai_china_ratio_yoy_pct,thai_gain_vs_baseline_musd,china_change_vs_baseline_musd,share_gain_vs_baseline,china_decline_positive_musd,baseline_thai,baseline_china,baseline_share,baseline_ratio,months_since_tariff,time_index,ym_fe
0,2015-01-01,2015,1,2015-01,39,Plastics,Materials,4,4,"1,173.325",38.941,"1,212.266",0.032,0.968,-3.381,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,-7.660,-111.401,-0.003,111.401,46.601,"1,284.727",0.035,-3.299,-42,1,2015-01
1,2015-02-01,2015,2,2015-02,39,Plastics,Materials,4,4,989.339,26.990,"1,016.329",0.027,0.973,-3.566,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,-19.610,-295.388,-0.009,295.388,46.601,"1,284.727",0.035,-3.299,-41,2,2015-02
2,2015-03-01,2015,3,2015-03,39,Plastics,Materials,4,4,"1,295.903",44.486,"1,340.389",0.033,0.967,-3.350,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,-2.114,11.176,-0.002,0.000,46.601,"1,284.727",0.035,-3.299,-40,3,2015-03
3,2015-04-01,2015,4,2015-04,39,Plastics,Materials,4,4,"1,149.515",53.463,"1,202.977",0.044,0.956,-3.050,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,6.862,-135.212,0.009,135.212,46.601,"1,284.727",0.035,-3.299,-39,4,2015-04
4,2015-05-01,2015,5,2015-05,39,Plastics,Materials,4,4,"1,291.836",46.439,"1,338.275",0.035,0.965,-3.305,Pre-tariff baseline,0,0.000,0.800,0.000,NaN,NaN,NaN,NaN,-0.162,7.109,-0.000,0.000,46.601,"1,284.727",0.035,-3.299,-38,5,2015-05


In [25]:
model_ratio = smf.ols(
    formula="""
    log_thai_china_ratio ~ tariff_intensity + C(hts2) + C(month) + time_index
    """,
    data=reg
).fit(cov_type="HC3")

print(model_ratio.summary())

                             OLS Regression Results                             
Dep. Variable:     log_thai_china_ratio   R-squared:                       0.927
Model:                              OLS   Adj. R-squared:                  0.926
Method:                   Least Squares   F-statistic:                     1114.
Date:                  Tue, 23 Jun 2026   Prob (F-statistic):               0.00
Time:                          05:37:02   Log-Likelihood:                -511.99
No. Observations:                  1452   AIC:                             1072.
Df Residuals:                      1428   BIC:                             1199.
Df Model:                            23                                         
Covariance Type:                    HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           

In [26]:
beta_ratio = model_ratio.params.get("tariff_intensity", np.nan)

effect_10pp = (np.exp(beta_ratio * 0.10) - 1) * 100
effect_25pp = (np.exp(beta_ratio * 0.25) - 1) * 100

print("Estimated coefficient on tariff intensity:", round(beta_ratio, 4))
print("Estimated effect of a 10 percentage-point tariff-intensity increase:")
print(round(effect_10pp, 2), "% change in Thailand-to-China import ratio")
print()
print("Estimated effect of a 25 percentage-point tariff-intensity increase:")
print(round(effect_25pp, 2), "% change in Thailand-to-China import ratio")

Estimated coefficient on tariff intensity: -0.2159
Estimated effect of a 10 percentage-point tariff-intensity increase:
-2.14 % change in Thailand-to-China import ratio

Estimated effect of a 25 percentage-point tariff-intensity increase:
-5.25 % change in Thailand-to-China import ratio


# 12. Dollar-Gain Model

The first regression estimates relative substitution. This second model estimates the relationship between tariff intensity and Thailand's monthly import gains in millions of dollars compared with the pre-tariff baseline.

In [27]:
model_dollars = smf.ols(
    formula="""
    thai_gain_vs_baseline_musd ~ tariff_intensity + C(hts2) + C(month) + time_index
    """,
    data=reg
).fit(cov_type="HC3")

print(model_dollars.summary())

                                OLS Regression Results                                
Dep. Variable:     thai_gain_vs_baseline_musd   R-squared:                       0.394
Model:                                    OLS   Adj. R-squared:                  0.384
Method:                         Least Squares   F-statistic:                     23.38
Date:                        Tue, 23 Jun 2026   Prob (F-statistic):           8.22e-83
Time:                                05:37:02   Log-Likelihood:                -10075.
No. Observations:                        1452   AIC:                         2.020e+04
Df Residuals:                            1428   BIC:                         2.033e+04
Df Model:                                  23                                         
Covariance Type:                          HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------

In [28]:
beta_dollars = model_dollars.params.get("tariff_intensity", np.nan)

print("Estimated monthly Thailand gain per 1.00 tariff-intensity unit:")
print(round(beta_dollars, 2), "million USD")
print()
print("Estimated monthly Thailand gain per 0.10 tariff-intensity increase:")
print(round(beta_dollars * 0.10, 2), "million USD")
print()
print("Estimated monthly Thailand gain per 0.25 tariff-intensity increase:")
print(round(beta_dollars * 0.25, 2), "million USD")

Estimated monthly Thailand gain per 1.00 tariff-intensity unit:
60.3 million USD

Estimated monthly Thailand gain per 0.10 tariff-intensity increase:
6.03 million USD

Estimated monthly Thailand gain per 0.25 tariff-intensity increase:
15.07 million USD


In [29]:
regression_summary = pd.DataFrame({
    "Model": [
        "Substitution ratio model",
        "Thailand dollar-gain model"
    ],
    "Dependent Variable": [
        "log(Thailand imports + 1) - log(China imports + 1)",
        "Thailand imports vs pre-tariff baseline, million USD"
    ],
    "Tariff Intensity Coefficient": [
        model_ratio.params.get("tariff_intensity", np.nan),
        model_dollars.params.get("tariff_intensity", np.nan)
    ],
    "Robust p-value": [
        model_ratio.pvalues.get("tariff_intensity", np.nan),
        model_dollars.pvalues.get("tariff_intensity", np.nan)
    ],
    "R-squared": [
        model_ratio.rsquared,
        model_dollars.rsquared
    ],
    "Observations": [
        int(model_ratio.nobs),
        int(model_dollars.nobs)
    ]
})

regression_summary

,Model,Dependent Variable,Tariff Intensity Coefficient,Robust p-value,R-squared,Observations
0,Substitution ratio model,log(Thailand imports + 1) - log(China imports ...,-0.216,0.236,0.927,1452
1,Thailand dollar-gain model,"Thailand imports vs pre-tariff baseline, milli...",60.298,0.625,0.394,1452


In [30]:
model_ratio_year_fe = smf.ols(
    formula="""
    log_thai_china_ratio ~ tariff_intensity + C(hts2) + C(month) + C(year)
    """,
    data=reg
).fit(cov_type="HC3")

print(model_ratio_year_fe.summary())

                             OLS Regression Results                             
Dep. Variable:     log_thai_china_ratio   R-squared:                       0.940
Model:                              OLS   Adj. R-squared:                  0.938
Method:                   Least Squares   F-statistic:                     978.9
Date:                  Tue, 23 Jun 2026   Prob (F-statistic):               0.00
Time:                          05:37:02   Log-Likelihood:                -372.01
No. Observations:                  1452   AIC:                             810.0
Df Residuals:                      1419   BIC:                             984.3
Df Model:                            32                                         
Covariance Type:                    HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           

## Why add year fixed effects?

Year fixed effects control for broad shocks that affect every sector in the same year, such as COVID-era supply chain disruption, inflation, demand changes, or global shipping stress.

If the tariff-intensity coefficient remains positive after adding year fixed effects, the substitution signal is more convincing.

# 13. Tariff Whack-a-Mole Simulator

The simulator estimates how much Thailand could gain in each sector under a new tariff shock.

It uses the regression model to translate tariff pressure into:

1. predicted percentage change in Thailand's position relative to China
2. predicted monthly dollar gain for Thailand

In [31]:
def simulate_tariff_shock(sector_name, new_tariff_rate=0.25):
    """
    Simulates Thailand's potential sector-level gain under a new tariff shock.

    sector_name: sector name as shown in the panel
    new_tariff_rate: decimal tariff rate, for example 0.25 for 25%
    """
    
    latest_date = panel["date"].max()
    
    latest = panel[
        (panel["sector"] == sector_name) &
        (panel["date"] == latest_date)
    ].copy()
    
    if latest.empty:
        return f"No data found for sector: {sector_name}"
    
    exposure = latest["exposure_score"].mean()
    current_thai = latest["thailand_imports_musd"].sum()
    current_china = latest["china_imports_musd"].sum()
    
    simulated_intensity = new_tariff_rate * exposure
    
    ratio_pct_change = (np.exp(beta_ratio * simulated_intensity) - 1) * 100
    estimated_monthly_gain = beta_dollars * simulated_intensity
    
    result = pd.DataFrame([{
        "Sector": sector_name,
        "Latest Date": latest_date.strftime("%Y-%m"),
        "Current Thailand Imports, Million USD": current_thai,
        "Current China Imports, Million USD": current_china,
        "Assumed New Tariff Rate": new_tariff_rate,
        "Sector Exposure Score": exposure,
        "Simulated Tariff Intensity": simulated_intensity,
        "Predicted Thailand-to-China Ratio Change %": ratio_pct_change,
        "Predicted Monthly Thailand Gain, Million USD": estimated_monthly_gain
    }])
    
    return result


simulate_tariff_shock("Electronics & electrical equipment", new_tariff_rate=0.25)

,Sector,Latest Date,"Current Thailand Imports, Million USD","Current China Imports, Million USD",Assumed New Tariff Rate,Sector Exposure Score,Simulated Tariff Intensity,Predicted Thailand-to-China Ratio Change %,"Predicted Monthly Thailand Gain, Million USD"
0,Electronics & electrical equipment,2025-12,"3,319.005","4,819.582",0.250,1.000,0.250,-5.254,15.074


In [32]:
simulation_outputs = []

for sector in sorted(panel["sector"].dropna().unique()):
    sim = simulate_tariff_shock(sector, new_tariff_rate=0.25)
    if isinstance(sim, pd.DataFrame):
        simulation_outputs.append(sim)

simulation_df = pd.concat(simulation_outputs, ignore_index=True)

simulation_df = simulation_df.sort_values(
    "Predicted Monthly Thailand Gain, Million USD",
    ascending=False
)

simulation_df

,Sector,Latest Date,"Current Thailand Imports, Million USD","Current China Imports, Million USD",Assumed New Tariff Rate,Sector Exposure Score,Simulated Tariff Intensity,Predicted Thailand-to-China Ratio Change %,"Predicted Monthly Thailand Gain, Million USD"
1,Electronics & electrical equipment,2025-12,"3,319.005","4,819.582",0.250,1.000,0.250,-5.254,15.074
5,Machinery & computers,2025-12,"4,304.473","2,682.514",0.250,1.000,0.250,-5.254,15.074
8,Rubber products,2025-12,391.035,109.850,0.250,0.900,0.225,-4.741,13.567
9,Vehicles & auto parts,2025-12,176.857,838.686,0.250,0.900,0.225,-4.741,13.567
0,Articles of steel,2025-12,120.933,516.044,0.250,0.800,0.200,-4.225,12.060
7,Plastics,2025-12,183.649,"1,029.993",0.250,0.800,0.200,-4.225,12.060
3,Furniture,2025-12,155.001,803.655,0.250,0.800,0.200,-4.225,12.060
6,Medical/precision instruments,2025-12,177.761,741.906,0.250,0.800,0.200,-4.225,12.060
2,Footwear,2025-12,11.348,354.452,0.250,0.600,0.150,-3.186,9.045
4,Knit apparel,2025-12,62.511,361.621,0.250,0.600,0.150,-3.186,9.045


In [33]:
fig = px.bar(
    simulation_df,
    x="Predicted Monthly Thailand Gain, Million USD",
    y="Sector",
    orientation="h",
    title="Simulator: Predicted Monthly Thailand Gain Under a 25% China Tariff Shock",
    labels={
        "Predicted Monthly Thailand Gain, Million USD": "Predicted Monthly Gain, Million USD",
        "Sector": "Sector"
    },
    hover_data=[
        "Current Thailand Imports, Million USD",
        "Current China Imports, Million USD",
        "Sector Exposure Score",
        "Predicted Thailand-to-China Ratio Change %"
    ]
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

# 14. Scenario Analysis

This section tests several possible tariff shock sizes. Instead of only analyzing the past, the notebook becomes a forward-looking policy simulator.

In [34]:
scenario_rates = [0.075, 0.10, 0.25, 0.50]

scenario_results = []

for rate in scenario_rates:
    for sector in sorted(panel["sector"].dropna().unique()):
        sim = simulate_tariff_shock(sector, new_tariff_rate=rate)
        if isinstance(sim, pd.DataFrame):
            sim["Scenario"] = f"{int(rate * 100)}% tariff shock"
            scenario_results.append(sim)

scenario_df = pd.concat(scenario_results, ignore_index=True)

scenario_df.head()

,Sector,Latest Date,"Current Thailand Imports, Million USD","Current China Imports, Million USD",Assumed New Tariff Rate,Sector Exposure Score,Simulated Tariff Intensity,Predicted Thailand-to-China Ratio Change %,"Predicted Monthly Thailand Gain, Million USD",Scenario
0,Articles of steel,2025-12,120.933,516.044,0.075,0.800,0.060,-1.287,3.618,7% tariff shock
1,Electronics & electrical equipment,2025-12,"3,319.005","4,819.582",0.075,1.000,0.075,-1.606,4.522,7% tariff shock
2,Footwear,2025-12,11.348,354.452,0.075,0.600,0.045,-0.967,2.713,7% tariff shock
3,Furniture,2025-12,155.001,803.655,0.075,0.800,0.060,-1.287,3.618,7% tariff shock
4,Knit apparel,2025-12,62.511,361.621,0.075,0.600,0.045,-0.967,2.713,7% tariff shock


In [35]:
fig = px.bar(
    scenario_df,
    x="Predicted Monthly Thailand Gain, Million USD",
    y="Sector",
    color="Scenario",
    barmode="group",
    orientation="h",
    title="Scenario Analysis: Thailand's Predicted Sector Gains Under Different Tariff Shocks",
    labels={
        "Predicted Monthly Thailand Gain, Million USD": "Predicted Monthly Gain, Million USD",
        "Sector": "Sector"
    }
)

fig.show()

In [36]:
def sector_card(sector_name):
    data = panel[panel["sector"] == sector_name].copy()
    
    pre = data[data["date"] < pd.Timestamp("2018-07-01")]
    post = data[data["date"] >= pd.Timestamp("2018-07-01")]
    
    pre_share = pre["thailand_share"].mean()
    post_share = post["thailand_share"].mean()
    share_change = post_share - pre_share
    
    pre_thai = pre["thailand_imports_musd"].mean()
    post_thai = post["thailand_imports_musd"].mean()
    thai_gain = post_thai - pre_thai
    
    pre_china = pre["china_imports_musd"].mean()
    post_china = post["china_imports_musd"].mean()
    china_change = post_china - pre_china
    
    print("=" * 80)
    print(sector_name.upper())
    print("=" * 80)
    print(f"Pre-tariff Thailand share:   {pre_share:.2%}")
    print(f"Post-tariff Thailand share:  {post_share:.2%}")
    print(f"Share change:                {share_change:.2%}")
    print()
    print(f"Average monthly Thai import gain:    ${thai_gain:,.2f}M")
    print(f"Average monthly China import change: ${china_change:,.2f}M")
    
    if share_change > 0 and thai_gain > 0:
        print("Signal: Thailand gained relative position in this sector.")
    elif share_change > 0:
        print("Signal: Thailand gained share, but dollar gains are mixed.")
    else:
        print("Signal: No strong Thailand substitution pattern.")
    print()


for sector in sorted(panel["sector"].dropna().unique()):
    sector_card(sector)

ARTICLES OF STEEL
Pre-tariff Thailand share:   5.31%
Post-tariff Thailand share:  8.91%
Share change:                3.60%

Average monthly Thai import gain:    $39.75M
Average monthly China import change: $42.82M
Signal: Thailand gained relative position in this sector.

ELECTRONICS & ELECTRICAL EQUIPMENT
Pre-tariff Thailand share:   5.61%
Post-tariff Thailand share:  12.92%
Share change:                7.31%

Average monthly Thai import gain:    $721.59M
Average monthly China import change: $-1,181.29M
Signal: Thailand gained relative position in this sector.

FOOTWEAR
Pre-tariff Thailand share:   0.71%
Post-tariff Thailand share:  1.21%
Share change:                0.50%

Average monthly Thai import gain:    $1.09M
Average monthly China import change: $-365.26M
Signal: Thailand gained relative position in this sector.

FURNITURE
Pre-tariff Thailand share:   0.87%
Post-tariff Thailand share:  4.68%
Share change:                3.82%

Average monthly Thai import gain:    $55.29M
Avera

In [37]:
top_sankey = leaderboard.head(6).copy()

labels = (
    ["U.S. tariff pressure on China"] +
    top_sankey["sector"].tolist() +
    ["Potential Thailand supply-chain gain"]
)

source = []
target = []
value = []

for _, row in top_sankey.iterrows():
    sector_index = labels.index(row["sector"])
    
    source.append(0)
    target.append(sector_index)
    value.append(max(row["whack_a_mole_index"], 0.01))
    
    source.append(sector_index)
    target.append(len(labels) - 1)
    value.append(max(row["avg_thai_gain"], 0.01))

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=18,
        label=labels
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    )
)])

fig.update_layout(
    title_text="Tariff Whack-a-Mole Flow: From China Shock to Thailand Sector Gains",
    font_size=11
)

fig.show()

# 15. Key Findings

## 1. Trade diversion is sector-specific

Thailand does not gain equally across every sector. Some sectors show stronger substitution signals than others.

This matters because supply chains only relocate when a country has enough production capacity, supplier networks, logistics infrastructure, and export readiness.

## 2. The Whack-a-Mole Index identifies the strongest substitution sectors

The Whack-a-Mole Index ranks sectors using Thailand's gains, China's decline, Thailand's import share increase, and sector exposure.

The strongest sectors are the ones where Thailand appears most likely to have benefited from supply-chain leakage.

## 3. The event study checks timing

The event study helps test whether Thailand's gains appeared after the 2018 tariff shock.

This is stronger than only comparing averages because timing matters in a trade diversion argument.

## 4. The regression model turns the story into a measurable relationship

The fixed-effects regression estimates whether tariff intensity is associated with Thailand gaining relative to China.

This creates a substitution semi-elasticity rather than only a visual story.

## 5. The simulator turns the notebook into a decision tool

The simulator estimates which Thai sectors may gain under future tariff shocks.

This makes the project useful for supply-chain risk analysis, economic policy discussion, and investment research.

# 16. Limitations

This notebook identifies trade diversion signals, not perfect causality.

Important limitations:

1. The data is HTS 2-digit sector-level, not HS6 or HS10 product-level.
2. The tariff variable is a sector-level proxy, not a perfect product-specific tariff match.
3. Thailand's export gains may reflect factors besides tariffs, including exchange rates, foreign direct investment, COVID disruptions, and changing U.S. demand.
4. Some trade may involve transshipment or rules-of-origin complexity that is not directly visible in this dataset.
5. The model uses sector exposure scores created for this project, so the results should be interpreted as a structured analytical framework rather than an official tariff classification.

A stronger future version would use HS6 or HS10 product-level data and match each product to exact Section 301 tariff-list exposure.

# 17. Conclusion

This notebook built a Tariff Whack-a-Mole Simulator to study how U.S.-China tariff pressure may have redirected supply-chain activity toward Thailand.

The project combined:

- official monthly U.S. import data
- HTS sector classification
- tariff shock timing
- trade diversion metrics
- event-study visualization
- fixed-effects regression
- a custom Whack-a-Mole Index
- a forward-looking tariff simulator

The main contribution is not simply showing that trade changed. The notebook creates a reusable framework for measuring how geopolitical shocks can redirect supply chains across countries.

For Thailand, the results highlight both opportunity and risk. Trade diversion can support manufacturing growth, but it can also expose Thailand to future scrutiny over supply-chain dependence, transshipment concerns, and rules-of-origin pressure.

In [38]:
print("Final project check")
print("-------------------")
print("Trade rows:", len(trade))
print("Panel rows:", len(panel))
print("Countries:", sorted(trade["country"].dropna().unique()))
print("Date range:", panel["date"].min().strftime("%Y-%m"), "to", panel["date"].max().strftime("%Y-%m"))
print("Number of sectors:", panel["sector"].nunique())
print("Leaderboard rows:", len(leaderboard))
print("Simulation rows:", len(simulation_df))

print("\nTop 5 Whack-a-Mole sectors:")
display(leaderboard[["sector", "whack_a_mole_index"]].head(5))

Final project check
-------------------
Trade rows: 2904
Panel rows: 1452
Countries: ['China', 'Thailand']
Date range: 2015-01 to 2025-12
Number of sectors: 11
Leaderboard rows: 11
Simulation rows: 11

Top 5 Whack-a-Mole sectors:


,sector,whack_a_mole_index
7,Electronics & electrical equipment,75.391
6,Machinery & computers,70.911
1,Rubber products,55.836
8,Vehicles & auto parts,23.600
10,Furniture,22.629
